In [1]:
import pandas as pd
df = pd.read_csv("students_21_raw.csv")
df.head(10)


,student id,Student Name,grade,subject,Test Date,homework score,QuizScore,Exam score,attendance %,notes
0,201,ali ahmed,7,math,2025-1-03,85,78,92,88,good progress
1,202,Aisha Ali,7,MATH,01/05/25,,81,89,95,missing homework last week
2,203,MOHAMED omar,8,Science,2025/01/07,90,,88,92,NaN
3,204,Sara Hassan,7,MATH,2025-1-9,72,69,,75,needs help with fractions
4,205,abdel rahman,8,Science,09-01-2025,88,84,91,,great participation
5,206,Lina Mahmoud,7,Math,2025-01-10,95,94,99,100,top of the class
6,207,Ali Ahmed,7,Math,03-01-2025,85,78,92,88,duplicate row?
7,208,aisha ali,7,Math,2025-1-5,80,81,89,95,NaN
8,209,Khalid Sami,9,PHYSICS,2025-01-11,,,76,60,low attendance
9,210,Farah Omar,8,science,2025-01-12,88,85,90,98,NaN


In [2]:

# 2. Standardize column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("%", "percent")
)

print("Columns after cleaning:")
print(df.columns.tolist())

# 3. Replace 'absent' in homework with NaN
df["homework_score"] = df["homework_score"].replace("absent", pd.NA)

# 4. Convert numeric columns
numeric_cols = ["homework_score", "quizscore", "exam_score", "attendance_percent"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[["student__name", "homework_score", "quizscore", "exam_score", "attendance_percent"]].head(10)


Columns after cleaning:
['student_id', 'student__name', 'grade', 'subject', 'test_date', 'homework_score', 'quizscore', 'exam_score', 'attendance_percent', 'notes']


,student__name,homework_score,quizscore,exam_score,attendance_percent
0,ali ahmed,85.0,78.0,92.0,88.0
1,Aisha Ali,NaN,81.0,89.0,95.0
2,MOHAMED omar,90.0,NaN,88.0,92.0
3,Sara Hassan,72.0,69.0,NaN,75.0
4,abdel rahman,88.0,84.0,91.0,NaN
5,Lina Mahmoud,95.0,94.0,99.0,100.0
6,Ali Ahmed,85.0,78.0,92.0,88.0
7,aisha ali,80.0,81.0,89.0,95.0
8,Khalid Sami,NaN,NaN,76.0,60.0
9,Farah Omar,88.0,85.0,90.0,98.0


In [3]:
# Strip spaces and turn to string
for col in df.select_dtypes(include="object"):
    df[col] = df[col].astype(str).str.strip()

# Fix student names -> Title Case
df["student__name"] = df["student__name"].str.title()

# Normalize subject
df["subject"] = df["subject"].str.lower().str.strip()
df["subject"] = df["subject"].replace({
    "math": "Math",
    "science": "Science",
    "physics": "Physics"
})


In [4]:
df[["student__name", "subject"]].head(10)


,student__name,subject
0,Ali Ahmed,Math
1,Aisha Ali,Math
2,Mohamed Omar,Science
3,Sara Hassan,Math
4,Abdel Rahman,Science
5,Lina Mahmoud,Math
6,Ali Ahmed,Math
7,Aisha Ali,Math
8,Khalid Sami,Physics
9,Farah Omar,Science


In [5]:
df["test_date"] = pd.to_datetime(
    df["test_date"],
    errors="coerce",
    dayfirst=False,           # most are like 2025-01-03 or 01/05/25
    infer_datetime_format=True
)

df[["test_date"]].head(10)


/tmp/ipython-input-3245883866.py:1: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df["test_date"] = pd.to_datetime(


,test_date
0,2025-01-03
1,NaT
2,NaT
3,2025-01-09
4,NaT
5,2025-01-10
6,NaT
7,2025-01-05
8,2025-01-11
9,2025-01-12


In [6]:
# Rename columns if needed
df = df.rename(columns={"homework_score": "homework", "QuizScore".lower(): "quizscore"}, errors="ignore")

numeric_cols = ["homework", "QuizScore", "Exam score", "attendance_percent"]

# In case of slight name differences:
for col in df.columns:
    print(col)


student_id
student__name
grade
subject
test_date
homework
quizscore
exam_score
attendance_percent
notes


In [7]:
# See all columns
df.columns


Index(['student_id', 'student__name', 'grade', 'subject', 'test_date',
       'homework', 'quizscore', 'exam_score', 'attendance_percent', 'notes'],
      dtype='object')

In [8]:
# 7. Remove exact duplicates based on student + test date + subject
before = len(df)
df = df.drop_duplicates(subset=["student_id", "test_date", "subject"])
after = len(df)

print("Rows before:", before)
print("Rows after :", after)
print("Removed    :", before - after)


Rows before: 21
Rows after : 21
Removed    : 0


In [9]:
df.columns.tolist()

['student_id',
 'student__name',
 'grade',
 'subject',
 'test_date',
 'homework',
 'quizscore',
 'exam_score',
 'attendance_percent',
 'notes']

In [10]:
# 8. Compute final_score (30% homework, 30% quiz, 40% exam)
df["final_score"] = (
    df["homework"].fillna(0)    * 0.3 +
    df["quizscore"].fillna(0)   * 0.3 +
    df["exam_score"].fillna(0)  * 0.4
)

# Risk flags
df["low_attendance"] = df["attendance_percent"] < 80
df["at_risk"] = (df["final_score"] < 70) | df["low_attendance"]

df[["student__name", "grade", "subject",
    "final_score", "attendance_percent", "at_risk"]].head(15)


,student__name,grade,subject,final_score,attendance_percent,at_risk
0,Ali Ahmed,7,Math,85.7,88.0,False
1,Aisha Ali,7,Math,59.9,95.0,True
2,Mohamed Omar,8,Science,62.2,92.0,True
3,Sara Hassan,7,Math,42.3,75.0,True
4,Abdel Rahman,8,Science,88.0,NaN,False
5,Lina Mahmoud,7,Math,96.3,100.0,False
6,Ali Ahmed,7,Math,85.7,88.0,False
7,Aisha Ali,7,Math,83.9,95.0,False
8,Khalid Sami,9,Physics,30.4,60.0,True
9,Farah Omar,8,Science,87.9,98.0,False


In [11]:
summary_by_student = (
    df.groupby("student__name")
      .agg(
          tests_taken     = ("test_date", "count"),
          avg_final_score = ("final_score", "mean"),
          avg_attendance  = ("attendance_percent", "mean"),
          any_risk        = ("at_risk", "max"),
      )
      .sort_values("avg_final_score", ascending=False)
)

summary_by_student


,tests_taken,avg_final_score,avg_attendance,any_risk
student__name,,,,
Lina Mahmoud,1,96.3,100.0,False
Zainab Ali,1,93.3,99.0,False
Abdel Rahman,1,88.0,90.0,False
Farah Omar,1,87.9,98.0,False
Ahmed Sami,0,87.6,93.0,False
Ali Ahmed,1,85.7,88.0,False
Ali Ahmed,2,85.7,88.0,False
Omar Ali,1,84.9,90.0,False
Aisha Ali,1,83.9,95.0,False


In [12]:
summary_by_subject = (
    df.groupby("subject")
      .agg(
          tests_taken     = ("test_date", "count"),
          avg_final_score = ("final_score", "mean"),
          avg_attendance  = ("attendance_percent", "mean"),
      )
)

summary_by_subject


,tests_taken,avg_final_score,avg_attendance
subject,,,
Math,9,76.415385,90.461538
Physics,1,59.000000,76.500000
Science,3,82.166667,91.600000


In [13]:
df["final_score"] = df["final_score"].round(1)   # or round(2) for two decimals


In [14]:
df["attendance_percent"] = df["attendance_percent"].round(0)


In [15]:
df.to_csv("students_21_cleaned.csv", index=False)
summary_by_student.to_csv("students_21_summary_by_student.csv")

print("Files saved!")


Files saved!


# New Section